In [ ]:
import os
import cv2
import numpy as np
from tqdm.auto import tqdm

# --- Configuration ---
DATA_DIR = '../cropped/' 
P, Q = 96, 84
NUM_CLASSES = 28
TRAIN_PER_CLASS = 19  
TEST_PER_CLASS = 12   

# --- Save directories ---
SAVE_DIR = './processed'
TRAIN_DIR = os.path.join(SAVE_DIR, 'train')
TEST_DIR = os.path.join(SAVE_DIR, 'test')

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Load OpenCV Haar Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


# ---------- CHECK IF DATA ALREADY EXISTS ----------
def is_data_saved():
    return len(os.listdir(TRAIN_DIR)) > 0 and len(os.listdir(TEST_DIR)) > 0


# ---------- LOAD SAVED DATA ----------
def load_saved_data():
    train_files = sorted(os.listdir(TRAIN_DIR))
    test_files = sorted(os.listdir(TEST_DIR))
    
    A_list, B_list = [], []
    train_labels, test_labels = [], []
    
    for f in train_files:
        A_list.append(np.load(os.path.join(TRAIN_DIR, f)))
        train_labels.append(int(f.split('_')[0].replace('class','')))
    
    for f in test_files:
        B_list.append(np.load(os.path.join(TEST_DIR, f)))
        test_labels.append(int(f.split('_')[0].replace('class','')))
    
    A_tensor = np.stack(A_list, axis=-1)
    A_vec = A_tensor.reshape((P * Q, A_tensor.shape[2]))
    
    return A_tensor, A_vec, np.array(train_labels), B_list, np.array(test_labels)


# ---------- ORIGINAL FUNCTION (WITH SAVING ADDED) ----------
def load_l2_normalized_eyb(data_dir):
    A_list, B_list = [], []
    train_labels, test_labels = [], []
    
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    classes = sorted(classes)[:NUM_CLASSES]
    
    print(f"🧹 Cleaning, Detecting, and Normalizing {len(classes)} classes...")

    for class_idx, class_name in enumerate(tqdm(classes, desc="Processing Classes")):
        class_path = os.path.join(data_dir, class_name)
        all_files = [f for f in os.listdir(class_path) if f.endswith(('.pgm', '.jpg', '.png'))]
        
        scored_images = []
        for img_name in all_files:
            if 'Ambient' in img_name: continue
            
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is not None:
                faces = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=5)
                
                if len(faces) > 0:
                    resized_img = cv2.resize(img, (Q, P)).astype(np.float64) / 255.0
                    
                    img_vec = resized_img.flatten()
                    norm_val = np.linalg.norm(img_vec)
                    
                    if norm_val > 1e-6:
                        img_vec = img_vec / norm_val
                        
                    normalized_img = img_vec.reshape((P, Q))
                    
                    brightness = np.mean(normalized_img)
                    if brightness > 0.002: 
                        scored_images.append((brightness, normalized_img))
        
        scored_images.sort(key=lambda x: x[0], reverse=True)
        
        if len(scored_images) < (TRAIN_PER_CLASS + TEST_PER_CLASS):
            print(f"\n⚠️ Warning: Class {class_name} only has {len(scored_images)} valid images.")
            continue
            
        # Training
        for i in range(TRAIN_PER_CLASS):
            A_list.append(scored_images[i][1])
            train_labels.append(class_idx)

            np.save(os.path.join(TRAIN_DIR, f"class{class_idx}_train_{i}.npy"),
                    scored_images[i][1])
            
        # Testing
        for i in range(TRAIN_PER_CLASS, TRAIN_PER_CLASS + TEST_PER_CLASS):
            B_list.append(scored_images[i][1])
            test_labels.append(class_idx)

            np.save(os.path.join(TEST_DIR, f"class{class_idx}_test_{i}.npy"),
                    scored_images[i][1])
            
    A_tensor = np.stack(A_list, axis=-1)
    A_vec = A_tensor.reshape((P * Q, A_tensor.shape[2]))
    
    return A_tensor, A_vec, np.array(train_labels), B_list, np.array(test_labels)


# ---------- MAIN EXECUTION ----------
if is_data_saved():
    print("⚡ Loading preprocessed data (skipping computation)...")
    A_tensor, A_vec, train_labels, B_list, test_labels = load_saved_data()
else:
    print("🚀 First run: Processing and saving data...")
    A_tensor, A_vec, train_labels, B_list, test_labels = load_l2_normalized_eyb(DATA_DIR)

print(f"\n✅ Data Ready!")
print(f"Training Tensor Shape: {A_tensor.shape}")
print(f"Testing List Size: {len(B_list)}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 8))
plt.suptitle("Optimized Training Basis (Top 19 Brightest Images)", fontsize=16)

for i in range(19):
    plt.subplot(4, 5, i + 1)
    plt.imshow(A_tensor[:, :,19+i], cmap='gray')
    plt.axis('off')
plt.show()

In [ ]:
import numpy as np
from scipy.linalg import svd

def prox_log_param(B_mat, lam, a=1.0):
    """
    Vectorized closed-form solution for the Proximal Operator of the 
    Parameterized Log-Sum Penalty using quadratic roots. 
    Runs instantly on matrices of any size.
    """
    b_abs = np.abs(B_mat)
    sign_b = np.sign(B_mat)
    
    A_quad = a
    B_quad = 1.0 - a * b_abs
    C_quad = lam - b_abs
    
    delta = B_quad**2 - 4 * A_quad * C_quad
    
    # Initialize output
    x_opt = np.zeros_like(b_abs)
    
    # Only process entries where the discriminant is valid (>= 0)
    valid = delta >= 0
    
    # Calculate the positive root for valid entries
    sqrt_delta = np.sqrt(np.maximum(delta, 0))
    x_root = (-B_quad + sqrt_delta) / (2 * A_quad)
    
    # The root must be strictly greater than 0
    valid = valid & (x_root > 0)
    
    # Evaluate the objective function to ensure the root is a global minimum
    obj_zero = 0.5 * (b_abs[valid])**2
    obj_root = (lam / a) * np.log(1.0 + a * x_root[valid]) + 0.5 * (x_root[valid] - b_abs[valid])**2
    
    # Keep roots that lower the objective compared to x=0
    better = obj_root < obj_zero
    
    final_valid = valid.copy()
    final_valid[valid] = better 
    
    x_opt[final_valid] = x_root[final_valid]
    
    return sign_b * x_opt

def generalized_svd_shrinkage_param(B_mat, lam, a):
    """
    Applies the Logarithmic proximal operator to the singular values of a matrix.
    This provides the necessary 'leash' to absorb massive block occlusions into L 
    without causing singular value leakage from the face.
    """
    U, S_vals, Vt = svd(B_mat, full_matrices=False)
    return U @ np.diag(prox_log_param(S_vals, lam, a)) @ Vt

def prox_arctan_param(B_mat, lam, a=1.0, max_iter=5, tol=1e-5):
    """
    Novel Proximal Operator for the Arctangent Penalty.
    Minimizes: 0.5 * (x - y)^2 + (lam / a) * arctan(a * |x|)
    Uses a highly optimized vectorized Newton-Raphson step to surgically isolate independent noise.
    """
    B_mat = np.array(B_mat, dtype=np.float64)
    b_abs = np.abs(B_mat)
    sign_b = np.sign(B_mat)
    
    A_sq = float(a)**2
    x_opt = np.zeros_like(b_abs)
    
    # Arctangent has a strict threshold. If the input is too small, 
    # it gets aggressively pushed to 0. 
    threshold = np.sqrt(2.0 * lam)
    valid_mask = b_abs > threshold
    
    if not np.any(valid_mask):
        return x_opt

    y_valid = b_abs[valid_mask]
    x_k = y_valid.copy() # Initialize at the upper bound
    
    # Vectorized Iterative Solver (Converges in ~3 iterations)
    for _ in range(max_iter):
        # f'(x) = x - y + penalty_gradient
        penalty_grad = (lam * a) / (1.0 + A_sq * (x_k**2))
        f_prime = x_k - y_valid + penalty_grad
        
        # f''(x) = 1 - penalty_second_derivative
        f_double_prime = 1.0 - (2.0 * lam * (a**3) * x_k) / ((1.0 + A_sq * (x_k**2))**2)
        f_double_prime = np.where(np.abs(f_double_prime) < 1e-8, 1e-8, f_double_prime)
        
        x_next = x_k - (f_prime / f_double_prime)
        x_next = np.maximum(x_next, 0.0) # Enforce non-negativity
        
        if np.max(np.abs(x_next - x_k)) < tol:
            x_k = x_next
            break
            
        x_k = x_next
        
    # Evaluate objective to ensure it's a global minimum compared to x=0
    obj_zero = 0.5 * (y_valid**2)
    obj_root = (lam / a) * np.arctan(a * x_k) + 0.5 * ((x_k - y_valid)**2)
    
    better_mask = obj_root < obj_zero
    
    # Map back to original shape
    final_indices = np.where(valid_mask)[0][better_mask]
    x_opt[valid_mask] = np.where(better_mask, x_k, 0.0)
    
    return sign_b * x_opt

def optimize_hybrid_ncrr_novel(A_tensor, A_vec, B_img, train_labels, num_classes, alpha, beta, a_param):
    """
    The main ADMM loop using the ASYMMETRIC HYBRID penalty framework.
    L (Block Occlusion) uses Logarithmic Shrinkage to control rank.
    S & v (Pixel Noise & Group Sparsity) use Arctangent Shrinkage for surgical sparsity.
    """
    p, q, N = A_tensor.shape
    B_vec = B_img.flatten()
    
    # 1. Calculate class-specific weights (w)
    distances = np.zeros(num_classes)
    for i in range(num_classes):
        A_i = A_vec[:, train_labels == i]
        beta_i = np.linalg.pinv(A_i.T @ A_i) @ A_i.T @ B_vec
        distances[i] = np.linalg.norm(B_vec - A_i @ beta_i, 2)
    w = (distances - np.min(distances)) / (np.max(distances) - np.min(distances) + 1e-8)
    
    A_T_A_inv = np.linalg.inv(A_vec.T @ A_vec + np.eye(N)) 
    
    # Initialize Matrices
    L, S = np.zeros((p, q)), np.zeros((p, q))
    z, u = np.zeros(N), np.zeros(N)
    j, u_tilde, v = np.zeros(num_classes), np.zeros(num_classes), np.zeros(num_classes)
    Y1, y2, y3, y4 = np.zeros((p, q)), np.zeros(N), np.zeros(num_classes), np.zeros(num_classes)
    
    mu, rho, tol, mu_max, max_iter = 1.0, 1.01, 1e-6, 1e6, 50
    Az = np.zeros((p, q)) 
    
    # ADMM Loop
    for _ in range(max_iter):
        
        # 1. THE LEASH: Update L (Low-Rank Error / Black Box) using LOG SVD Shrinkage
        L = generalized_svd_shrinkage_param(B_img - Az - S + Y1 / mu, 1.0 / mu, a_param)
        
        # 2. THE SCALPEL: Update S (Sparse Error / Pixel Noise) using ARCTAN Operator
        S = prox_arctan_param(B_img - Az - L + Y1 / mu, alpha / mu, a_param)
        
        # Update j
        j = (u_tilde + w * v + (y3 + w * y4) / mu) / (np.ones(num_classes) + w * w)
        
        # Update z (Coding Coefficients)
        h4_mat = B_img - S - L + Y1 / mu
        z = A_T_A_inv @ (A_vec.T @ h4_mat.flatten() + u - y2 / mu)
        
        # Update u (Group Sparsity Math)
        for i in range(num_classes):
            idx = (train_labels == i)
            lam_u = (y3[i] / mu) - j[i]
            
            A_vec_i = z[idx] + y2[idx] / mu
            norm_A = np.linalg.norm(A_vec_i, 2)
            
            if norm_A > 0:
                u[idx] = 0.5 * max(1 - lam_u / norm_A, 0) * A_vec_i 
            else:
                u[idx] = np.zeros_like(A_vec_i)
                
            u_tilde[i] = np.linalg.norm(u[idx], 2)
            
        # 3. IDENTITY SELECTION: Update v using ARCTAN Operator
        v = prox_arctan_param(w * j - y4 / mu, beta / mu, a_param)
        
        # Update Az
        Az = np.tensordot(A_tensor, z, axes=([2], [0]))
        err_eq1 = B_img - Az - L - S
        
        # Dual Variables Updates
        Y1 += mu * err_eq1
        y2 += mu * (z - u)
        y3 += mu * (u_tilde - j)
        y4 += mu * (v - w * j)
        
        mu = min(mu * rho, mu_max)
        
        # Check Convergence
        if (np.max(np.abs(err_eq1)) < tol and np.max(np.abs(z - u)) < tol and 
            np.max(np.abs(u_tilde - j)) < tol and np.max(np.abs(v - w * j)) < tol):
            break
            
    return z

In [ ]:
def classify(A_tensor, z_opt, train_labels, num_classes):
    residuals = np.zeros(num_classes)
    A_z_star = np.tensordot(A_tensor, z_opt, axes=([2], [0]))
    for i in range(num_classes):
        z_i = np.zeros_like(z_opt)
        z_i[train_labels == i] = z_opt[train_labels == i]
        A_i_z = np.tensordot(A_tensor, z_i, axes=([2], [0]))
        _, S_diff, _ = svd(A_z_star - A_i_z, full_matrices=False)
        residuals[i] = np.sum(np.log(np.abs(S_diff) + 1))
    return np.argmin(residuals)

def add_black_block(image, ratio):
    img_h, img_w = image.shape
    block_side = int(np.sqrt(img_h * img_w * ratio))
    block_side = min(block_side, img_h, img_w)
    x = random.randint(0, img_w - block_side)
    y = random.randint(0, img_h - block_side)
    occ_img = image.copy()
    occ_img[y:y+block_side, x:x+block_side] = 0.0 
    return occ_img

In [ ]:
import time
import itertools
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# --- EXPERIMENT SETUP ---
SUBSET_SIZE = 336 
VAL_SIZE = 15

test_indices = np.random.choice(len(B_list), size=SUBSET_SIZE, replace=False)
B_subset = [B_list[i] for i in test_indices]
labels_subset = test_labels[test_indices]


# alphas = [0.001, 0.005, 0.01]
# betas  = [0.001, 0.005, 0.01]
# a_params = [1.0, 10.0, 50.0, 100.0]

alphas = [0.005]
betas  = [0.001]
a_params = [50.0]


combinations = list(itertools.product(alphas, betas, a_params))
occlusion_ratios = [0.40]


results_per_ratio = []

print("🔬 HYBRID(LOG+TAN) LIVE VALIDATION DASHBOARD\n")

# ===================== MAIN LOOP =====================
for ratio in tqdm(occlusion_ratios, desc="🌐 Overall Progress"):

    # -------- Stratified Validation Selection --------
    unique_classes, unique_indices = np.unique(labels_subset, return_index=True)

    if len(unique_classes) >= VAL_SIZE:
        val_idx = np.random.choice(unique_indices, size=VAL_SIZE, replace=False)
    else:
        val_idx = np.random.choice(SUBSET_SIZE, size=VAL_SIZE, replace=False)

    best_acc, best_alpha, best_beta, best_a = -1, None, None, None

    # ===================== TUNING =====================
    param_bar = tqdm(combinations,
                     desc=f"⚙️ Tuning {int(ratio*100)}%",
                     leave=False)

    for alpha, beta, a_param in param_bar:

        correct = 0

        val_bar = tqdm(val_idx,
                       desc="   🧠 Validating",
                       leave=False)

        for i, idx in enumerate(val_bar, 1):

            img, true_label = B_subset[idx], labels_subset[idx]
            occ_img = add_black_block(img, ratio)

            z_opt = optimize_hybrid_ncrr_novel(
                A_tensor, A_vec, occ_img,
                train_labels, NUM_CLASSES,
                alpha, beta, a_param
            )

            pred_label = classify(
                A_tensor, z_opt,
                train_labels, NUM_CLASSES
            )

            if pred_label == true_label:
                correct += 1

            live_acc = (correct / i) * 100

            val_bar.set_postfix({
                "α": alpha,
                "β": beta,
                "a": a_param,
                "LiveAcc": f"{live_acc:.1f}%"
            })

        acc = correct / VAL_SIZE

        if acc > best_acc:
            best_acc = acc
            best_alpha = alpha
            best_beta = beta
            best_a = a_param

        param_bar.set_postfix({
            "BestAcc": f"{best_acc*100:.0f}%",
            "Best α": best_alpha,
            "Best β": best_beta,
            "Best a": best_a
        })
        
        # --- EARLY BREAK LOGIC ---
        if best_acc == 1.0:

            tqdm.write(f"⚡ Perfect validation achieved! Breaking early to save time.")
            break 
            
   
    param_bar.close()

    # ===================== TESTING =====================
    correct_predictions = 0

    test_bar = tqdm(zip(B_subset, labels_subset),
                    total=SUBSET_SIZE,
                    desc=f"🚀 Testing {int(ratio*100)}%",
                    leave=False)

    for i, (img, true_label) in enumerate(test_bar, 1):

        occ_img = add_black_block(img, ratio)

        z_opt = optimize_hybrid_ncrr_novel(
            A_tensor, A_vec, occ_img,
            train_labels, NUM_CLASSES,
            best_alpha, best_beta, best_a
        )

        pred_label = classify(
            A_tensor, z_opt,
            train_labels, NUM_CLASSES
        )

        if pred_label == true_label:
            correct_predictions += 1

        live_test_acc = (correct_predictions / i) * 100

        test_bar.set_postfix({
            "LiveAcc": f"{live_test_acc:.1f}%"
        })

    final_acc = (correct_predictions / SUBSET_SIZE) * 100

    results_per_ratio.append({
        "ratio": ratio,
        "accuracy": final_acc,
        "alpha": best_alpha,
        "beta": best_beta,
        "a": best_a
    })

    # -------- Summary Block --------
    print("\n========================================")
    print(f"Occlusion Level : {int(ratio*100)}%")
    print(f"Final Accuracy  : {final_acc:.2f}%")
    print(f"Best Parameters : α={best_alpha}, β={best_beta}, a={best_a}")
    print("========================================\n")

print("✅ All stages complete.")